# Cross-project prediction of Java code smells (SmellyCode++)

This notebook reproduces the experiment in the paper *Cross-Project Prediction of Java Code Smells with Classical Machine Learning Models: A Leakage-Aware Evaluation on SmellyCode++*.

1. Run the first cell; it clones the code from GitHub (or asks you to upload the code zip if cloning fails).
2. Run the cells in order. A CPU runtime is enough, and the whole run takes about 95 minutes on two CPU cores, most of it the two tuning steps.

Your outputs go to `work/`, and the outputs of the reported run are in `results/` for comparison.

In [ ]:
import os, zipfile, subprocess
REPO = 'https://github.com/Centia-0113/22287322_smellycode-xproject_code'
if not os.path.exists('run_all.sh'):
    if not os.path.isdir('22287322_smellycode-xproject_code'):
        if subprocess.call(['git', 'clone', '--depth', '1', REPO]) != 0:      # fallback: upload the code zip instead
            from google.colab import files
            up = files.upload()
            zipfile.ZipFile(next(iter(up))).extractall('.')
    for d in ['22287322_smellycode-xproject_code', 'smellycode-xproject']:
        if os.path.isdir(d): os.chdir(d); break
print(os.getcwd(), sorted(os.listdir('.')))


## Install the pinned library versions

These are the versions used for the reported run. If a pin cannot be installed on Colab's Python version, install the same packages without the version numbers. The results may then differ slightly.

In [ ]:
!pip install -q -r requirements.txt

## Download SmellyCode++ (618 MB, CC0 1.0)

If the download fails, download `multi-smell-dataset-v1_2.csv` yourself from https://doi.org/10.6084/m9.figshare.28519385 and put it in `data/`.

In [ ]:
import hashlib, urllib.request
os.makedirs('data', exist_ok=True)
CSV = 'data/multi-smell-dataset-v1_2.csv'
if not os.path.exists(CSV):
    urllib.request.urlretrieve('https://ndownloader.figshare.com/files/52714583', CSV)
h = hashlib.sha256()
with open(CSV, 'rb') as f:
    for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
print(os.path.getsize(CSV), 'bytes', h.hexdigest())
assert h.hexdigest() == '971a81da3f55326a5a56bae59c3cde34c4ebab1a99615ac0c474ce5ecf5029d2', 'not the file used in the paper'

## Run the pipeline

Steps 1–2 clean the data and make the seeded project-level split. Step 3 tunes on the training projects only. Step 4 scores the frozen models once on the test projects.

In [ ]:
!python src/step1_clean_group.py data/multi-smell-dataset-v1_2.csv work
!python src/step2_split.py work
!python src/clusters.py work | tail -12

In [ ]:
!python src/step3_tune.py work

In [ ]:
!python src/step4_test.py work > work/step4.log && tail -1 work/step4.log
!python src/step5_contrast.py work data/multi-smell-dataset-v1_2.csv
!python src/step6_facts.py data/multi-smell-dataset-v1_2.csv work
!python src/cleaning_records.py data/multi-smell-dataset-v1_2.csv work > work/cleaning_records.log && echo 'cleaning records written to work/cleaning_records/'

## Post hoc, exploratory analysis

These steps restrict each smell to its own granularity: God Class and Data Class use class-level samples, and Long Method and Feature Envy use method-level samples. The step after that re-scores the **previously examined** test projects, so its test results are not an independent test. Two further steps check the granularity rule against a Java parser and report the bootstrap details (resamples without positives, paired model differences). A population check re-scores the frozen primary models on the matched populations, which separates the change of evaluation population from the effect of training on matched samples. A further step checks the 873 samples the parser could not classify: it re-scores the matched models without them and repeats the matched analysis without them (in `work/unverified/`). The last step checks how sensitive the results are to the cleaning rule.

In [ ]:
!python src/granularity.py work | tail -6
!python src/posthoc_tune.py work | tail -3
!python src/posthoc_test.py work > work/posthoc_test.log && tail -1 work/posthoc_test.log
!python src/validate_granularity.py work > work/granularity_validation.log && head -12 work/granularity_validation.log
!python src/bootstrap_details.py work data/multi-smell-dataset-v1_2.csv > work/bootstrap_details.log && echo 'bootstrap details written'
!python src/population_check.py work > work/population_check.log && echo 'population check written'
!python src/unparsed_check.py --prepare work work/unverified
!python src/posthoc_tune.py work/unverified > work/unverified/posthoc_tune.log && tail -1 work/unverified/posthoc_tune.log
!python src/posthoc_test.py work/unverified > work/unverified/posthoc_test.log && tail -1 work/unverified/posthoc_test.log
!python src/unparsed_check.py work work/unverified > work/unparsed_check.log && echo 'unparsed-sample check written'
!python src/cleaning_sensitivity.py data/multi-smell-dataset-v1_2.csv work | tail -2
!python src/fig3.py work/results.json work/posthoc_results.json work/fig3.png

## Key results

The next cell prints the test results per smell (Tables III and IV) and compares your run with the reported one.

In [ ]:
import json, pandas as pd
R = json.load(open('work/results.json')); ref = json.load(open('results/results.json'))
rows = []
for m in ['RF', 'LR', 'XGB']:
    for s, v in R['test'][m]['per_smell'].items():
        rows.append({'model': m, 'smell': s, **{k: round(x, 3) for k, x in v.items()}})
    rows.append({'model': m, 'smell': 'macro', 'F1': round(R['test'][m]['macroF1'], 3), 'PRAUC': round(R['test'][m]['macroAP'], 3)})
print('best model by validation:', R['best_model_by_validation'])
display(pd.DataFrame(rows))
same = all(abs(R['test'][m]['macroF1'] - ref['test'][m]['macroF1']) < 1e-9 for m in ['RF', 'LR', 'XGB'])
print('macro-F1 identical to the reported run:', same)

In [ ]:
H = json.load(open('work/posthoc_results.json'))
rows = [{'model': m, 'smell': s, 'F1': round(v['F1'], 3), 'PR-AUC': round(v['PRAUC'], 3)} for m in ['RF', 'LR', 'XGB'] for s, v in H['test_rescored'][m]['per_smell'].items()]
print('post hoc (exploratory) - best model by matched validation:', H['best_model_by_validation'])
display(pd.DataFrame(rows))

In [ ]:
from IPython.display import Image
Image('work/fig3.png', width=520)

## Compare your run with the reported run

Every output in `work/` is compared with `results/` (numbers to a relative tolerance of 1e-9; run times are ignored).

In [ ]:
!python src/compare_outputs.py work results | tail -25